# Chapter 21: practical portfolio construction

Fresh hand checks use the shared core for covariance shrinkage, Euler risk allocation/ERC, Black–Litterman, binary Kelly versus a quadratic proxy, neutrality/projection and drift/turnover. The French/Project 3 gallery is skipped. RUN_DIR is allocated, but no new result files or figures are saved.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch21_portfolio_construction_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys
import numpy as np
from IPython.display import Image, display

sys.path.insert(0, str(ROOT / 'src'))

from quant_course.portfolio_construction import (
    ledoit_wolf_identity, risk_contributions, risk_budget_weights,
    max_sharpe_long_only, black_litterman, he_litterman_view_covariance,
    kelly_binary_fraction, binary_log_growth, gaussian_kelly_weights,
    quadratic_log_growth, exposure_report, project_to_constraints,
    drift_weights, turnover,
)
from quant_course.optimization_tools import solve_mean_variance, OptimizationError
print('Project root:', ROOT)

## 1. Ledoit–Wolf shrinkage

Use the centered $4\times2$ matrix $[[3,2],[-1,-2],[2,3],[-4,-3]]$ in percentage units. Check target scale 7, squared deviation 42.5, estimated error 8 and intensity 16/85. These covariance calculations do not establish performance.

In [ ]:
x_hand = np.array([[3.0, 2.0], [-1.0, -2.0], [2.0, 3.0], [-4.0, -3.0]])
lw = ledoit_wolf_identity(x_hand)

assert np.isclose(lw.target_scale, 7.0)
assert np.isclose(lw.intensity, 8.0 / 42.5)
np.testing.assert_allclose(lw.sample_covariance, [[7.5, 6.5], [6.5, 6.5]], atol=1e-12)

# 预期收缩矩阵
expected_shrunk = np.array([[7.40588235, 5.27647059], [5.27647059, 6.59411765]])
np.testing.assert_allclose(lw.covariance, expected_shrunk, atol=1e-6)

# 相关系数从 0.931 收缩到 0.755
corr_sample = lw.sample_covariance[0, 1] / np.sqrt(lw.sample_covariance[0, 0] * lw.sample_covariance[1, 1])
corr_shrunk = lw.covariance[0, 1] / np.sqrt(lw.covariance[0, 0] * lw.covariance[1, 1])
print(f'Sample correlation: {corr_sample:.4f}, Shrunk correlation: {corr_shrunk:.4f}')
assert corr_shrunk < corr_sample

## 2. Four-asset Euler decomposition and ERC

Volatilities are 10%, 20%, 30% and 40%, with the stated positive/negative correlation blocks. Equal weights give 11.51% volatility, with 47.17% of risk from the highest-volatility asset. Long-only minimum variance gives 8.63%; ERC gives 10.29% and 25% risk each. Equal marginal risk applies to held assets at the stated optimum, not automatically to omitted assets.

In [ ]:
vol_mrt = np.array([0.10, 0.20, 0.30, 0.40])
corr_mrt = np.array([
    [1.0, 0.8, 0.0, 0.0],
    [0.8, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, -0.5],
    [0.0, 0.0, -0.5, 1.0],
])
cov_mrt = np.outer(vol_mrt, vol_mrt) * corr_mrt

# 1/N 等权组合
ew_res = risk_contributions(np.full(4, 0.25), cov_mrt)
assert np.isclose(ew_res.volatility, 0.1151086, atol=1e-4)
np.testing.assert_allclose(ew_res.fractions, [0.123, 0.264, 0.141, 0.472], atol=1e-3)

# 长仓最小方差
mv_weights = solve_mean_variance(np.zeros(4), cov_mrt).weights
mv_res = risk_contributions(mv_weights, cov_mrt)
assert np.isclose(mv_res.volatility, 0.086303, atol=1e-4)
np.testing.assert_allclose(mv_weights, [0.7448, 0.0, 0.1517, 0.1034], atol=1e-3)

# 等风险贡献 (ERC)
erc_weights = risk_budget_weights(cov_mrt).weights
erc_res = risk_contributions(erc_weights, cov_mrt)
assert np.isclose(erc_res.volatility, 0.102934, atol=1e-4)
np.testing.assert_allclose(erc_weights, [0.3836, 0.1918, 0.2426, 0.1820], atol=1e-3)
np.testing.assert_allclose(erc_res.fractions, 0.25, atol=1e-6)

# 单调性定理验证
assert mv_res.volatility <= erc_res.volatility <= ew_res.volatility
print('MRT 四资产风险分解与 ERC 权重完全复核通过！')

## 3. Black–Litterman

For benchmark (0.6,0.4), risk aversion 2.5 and tau=0.05, a 10% annual excess-return view on asset 1 uses $\Omega=\tau P\Sigma P^\top=0.002$. Posterior means (9%,12.5%) imply unconstrained risky weights (0.7,0.4). Asset 2's mean rises while its weight stays fixed. Risky weights need not sum to one; cash financing is separate.

In [ ]:
cov_bl = np.array([[0.04, 0.02], [0.02, 0.09]])
w_mkt = np.array([0.6, 0.4])
p_view = np.array([[1.0, 0.0]])
q_view = np.array([0.10])
tau_val = 0.05
omega_view = he_litterman_view_covariance(p_view, cov_bl, tau=tau_val)

bl = black_litterman(
    cov_bl, w_mkt, risk_aversion=2.5, tau=tau_val,
    view_matrix=p_view, view_returns=q_view, view_covariance=omega_view,
)
np.testing.assert_allclose(bl.equilibrium_returns, [0.08, 0.12], atol=1e-12)
np.testing.assert_allclose(bl.posterior_mean, [0.09, 0.125], atol=1e-12)
np.testing.assert_allclose(bl.weights, [0.7, 0.4], atol=1e-12)
print('BL 两资产后验均值与权重完全匹配！')

## 4. Binary Kelly and a quadratic growth proxy

With success probability 0.55 and unit net odds, full Kelly fraction 0.10 gives growth 0.005008; half gives 0.003753, approximately 75% in this example, not an exact general binary identity. Double Kelly has negative expected log growth; under IID repetition wealth decays asymptotically. The separate quadratic proxy with mean 0.006 and variance 0.0016 gives weight 3.75; half keeps 75% proxy growth and 25% variance. This is not exact Gaussian simple-return log utility.

In [ ]:
assert np.isclose(kelly_binary_fraction(0.55, 1.0), 0.10)
g_full = binary_log_growth(0.10, 0.55, 1.0)
g_half = binary_log_growth(0.05, 0.55, 1.0)
g_double = binary_log_growth(0.20, 0.55, 1.0)

assert np.isclose(g_full, 0.0050083668, atol=1e-6)
assert np.isclose(g_half, 0.0037526078, atol=1e-6)
assert g_double < 0
print(f'Full Kelly growth: {g_full:.6f}, Half Kelly: {g_half:.6f}, Double Kelly: {g_double:.6f}')

# 连续二次近似
m_val = np.array([0.006])
cov_val = np.array([[0.0016]])
full_w = gaussian_kelly_weights(m_val, cov_val, fraction=1.0)
half_w = gaussian_kelly_weights(m_val, cov_val, fraction=0.5)
assert np.isclose(full_w[0], 3.75)
g_q_full = quadratic_log_growth(full_w, m_val, cov_val)
g_q_half = quadratic_log_growth(half_w, m_val, cov_val)
assert np.isclose(g_q_half, 0.75 * g_q_full)
print('Kelly 增长率权衡验证通过！')

## 5. Neutrality, projection and turnover

Weights (0.5,0.3,−0.4,−0.4) have zero signed net funding but beta 0.20 and nonzero sector exposures. Euclidean projection satisfies the specified constraints while reducing gross exposure from 1.60 to 0.20. Equal weights drift to (0.55,0.45) after returns (+10%,−10%); buys-plus-sells rebalancing turnover is 0.10.

In [ ]:
w_demo = np.array([0.5, 0.3, -0.4, -0.4])
beta_demo = np.array([1.2, 0.8, 1.0, 0.6])
sectors_demo = ['A', 'B', 'A', 'B']

rep = exposure_report(w_demo, beta_demo, sectors_demo)
assert np.isclose(rep.net, 0.0)
assert np.isclose(rep.beta, 0.2)
assert np.isclose(rep.sector_net['A'], 0.1)
assert np.isclose(rep.sector_net['B'], -0.1)

a_mat = np.vstack([np.ones(4), beta_demo, [1.0, 0.0, 1.0, 0.0]])
proj_w = project_to_constraints(w_demo, a_mat)
np.testing.assert_allclose(a_mat @ proj_w, 0.0, atol=1e-12)
print('投影后总暴露从 1.6 缩减至:', float(np.abs(proj_w).sum()))

# 漂移与换手
w_init = np.array([0.5, 0.5])
r_month = np.array([0.10, -0.10])
w_drift = drift_weights(w_init, r_month)
np.testing.assert_allclose(w_drift, [0.55, 0.45], atol=1e-12)
assert np.isclose(turnover(w_drift, w_init), 0.10, atol=1e-12)
print('漂移与换手台账核查通过！')

## 6. Historical gallery skipped

The next cell prints a skip message. No French or Project 3 historical charts or returns are reproduced.

In [ ]:
print("SKIPPED historical figure gallery; calculations above are freshly constructed, historical French/Project3 figures are not reproduced")


## Boundary examples

Complete signed sector neutrality implies dollar neutrality, not beta neutrality. The discrete second-order log expansion includes the squared mean: mean−(variance+mean²)/2. The variance-only quadratic proxy omits that term and does not establish existence of exact Gaussian simple-return log utility. Chinese success messages refer to the adjacent constructed assertions only.

In [ ]:
import numpy as np
w = np.array([0.5, -0.5])
sector = np.ones((1, 2))
beta = np.array([1.5, 0.5])
assert np.allclose(sector @ w, 0) and np.isclose(w.sum(), 0)
assert np.isclose(beta @ w, 0.5)
r = np.array([0.1, 0.2])  # 构造的等概率单期简单收益
full_second_order = r.mean() - 0.5 * (r.var() + r.mean() ** 2)
variance_only_proxy = r.mean() - 0.5 * r.var()
assert np.isclose(full_second_order, 0.1375)
assert np.isclose(variance_only_proxy, 0.14875)
print({"industry_neutral_beta_exposure": beta @ w, "exact_expected_log": np.log1p(r).mean(), "full_second_order": full_second_order, "variance_only_proxy": variance_only_proxy})